# Notebook 03: Exploratory Data Analysis (EDA)

## Objectives
- Explore the distribution of the Churn target variable
- Identify numeric features most correlated with Churn via heatmap
- Visualise the relationship between the top correlated features and Churn
- Examine key categorical features and their association with Churn
- Save all plots to `outputs/eda/` for use in the Streamlit dashboard

## Inputs
- `outputs/datasets/collection/telco_churn_cleaned.csv`

## Outputs
- `outputs/eda/churn_distribution.png`
- `outputs/eda/correlation_heatmap.png`
- `outputs/eda/tenure_vs_churn.png`
- `outputs/eda/monthlycharges_vs_churn.png`
- `outputs/eda/contract_vs_churn.png`
- `outputs/eda/techsupport_vs_churn.png`
- `outputs/eda/internetservice_vs_churn.png`

---

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.chdir('/workspace/customer-churn-predictor')  # adjust if running locally

os.makedirs('outputs/eda', exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print("Working directory:", os.getcwd())

## 2. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('outputs/datasets/collection/telco_churn_cleaned.csv')
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

## 3. Churn Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Churn'].value_counts()
ax.bar(['No Churn (0)', 'Churn (1)'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Churn Distribution')
ax.set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f'{v} ({v/len(df)*100:.1f}%)', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/churn_distribution.png', dpi=150)
plt.show()
print("Saved: churn_distribution.png")

**Interpretation**: The dataset has a class imbalance with approximately 73.5% non-churners and 26.5% churners. This imbalance will be addressed during modelling using SMOTE oversampling to avoid a biased classifier.

## 4. Correlation Heatmap (Numeric Features vs Churn)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax
)
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('outputs/eda/correlation_heatmap.png', dpi=150)
plt.show()

print("\nCorrelation with Churn (descending):")
print(corr_matrix['Churn'].drop('Churn').sort_values(ascending=False))
print("\nSaved: correlation_heatmap.png")

**Interpretation**: `tenure` has the strongest negative correlation with Churn (-0.35), meaning longer-serving customers are far less likely to leave. `MonthlyCharges` has a positive correlation (+0.19), indicating that customers paying more per month carry higher churn risk. `TotalCharges` is negatively correlated partly because it co-varies with tenure.

## 5. Top Numeric Features vs Churn

In [ ]:
# tenure vs Churn
fig, ax = plt.subplots(figsize=(7, 4))
df.groupby('Churn')['tenure'].plot(kind='hist', alpha=0.6, bins=30, ax=ax, legend=True)
ax.set_title('Tenure Distribution by Churn Status')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Count')
ax.legend(['No Churn', 'Churn'])
plt.tight_layout()
plt.savefig('outputs/eda/tenure_vs_churn.png', dpi=150)
plt.show()
print("Saved: tenure_vs_churn.png")

**Interpretation**: Churned customers are heavily concentrated at low tenure values, while retained customers are more evenly spread across all tenure levels. This confirms that new customers are at the greatest risk of churning and should be the focus of early retention efforts.

In [ ]:
# MonthlyCharges vs Churn
fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in df.groupby('Churn'):
    grp['MonthlyCharges'].plot(kind='hist', alpha=0.6, bins=30, ax=ax,
                               label='Churn' if label == 1 else 'No Churn')
ax.set_title('Monthly Charges Distribution by Churn Status')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda/monthlycharges_vs_churn.png', dpi=150)
plt.show()
print("Saved: monthlycharges_vs_churn.png")

**Interpretation**: The distribution of MonthlyCharges for churned customers is shifted towards higher values compared to retained customers. Customers paying more than ~$65/month show a noticeably elevated churn rate, suggesting price sensitivity is a key factor.

## 6. Key Categorical Features vs Churn

In [ ]:
# Contract vs Churn
fig, ax = plt.subplots(figsize=(7, 4))
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
ax.bar(contract_churn.index, contract_churn.values * 100, color='tomato')
ax.set_title('Churn Rate by Contract Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 100)
for i, v in enumerate(contract_churn.values):
    ax.text(i, v * 100 + 1, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/contract_vs_churn.png', dpi=150)
plt.show()
print("Saved: contract_vs_churn.png")

**Interpretation**: Month-to-month customers churn at approximately 43% compared to just 11% for one-year and 3% for two-year contract holders. Long-term contracts are the single most powerful retention instrument visible in this dataset.

In [ ]:
# TechSupport vs Churn
fig, ax = plt.subplots(figsize=(7, 4))
ts_churn = df.groupby('TechSupport')['Churn'].mean().sort_values(ascending=False)
ax.bar(ts_churn.index, ts_churn.values * 100, color='steelblue')
ax.set_title('Churn Rate by Tech Support Status')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 60)
for i, v in enumerate(ts_churn.values):
    ax.text(i, v * 100 + 1, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/techsupport_vs_churn.png', dpi=150)
plt.show()
print("Saved: techsupport_vs_churn.png")

**Interpretation**: Customers without tech support churn at roughly twice the rate of those who have it. Offering or promoting tech support, particularly to high-risk segments, could meaningfully reduce churn.

In [ ]:
# InternetService vs Churn
fig, ax = plt.subplots(figsize=(7, 4))
is_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
ax.bar(is_churn.index, is_churn.values * 100, color='mediumpurple')
ax.set_title('Churn Rate by Internet Service Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 60)
for i, v in enumerate(is_churn.values):
    ax.text(i, v * 100 + 1, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/internetservice_vs_churn.png', dpi=150)
plt.show()
print("Saved: internetservice_vs_churn.png")

**Interpretation**: Fibre optic internet customers churn at approximately 42%, compared to 19% for DSL customers and 7% for those with no internet service. This may reflect higher expectations or pricing concerns among fibre customers.

---
## Conclusions
- **Tenure** is the strongest numeric predictor of churn — low-tenure customers are at highest risk.
- **MonthlyCharges** is positively associated with churn — higher-paying customers are more likely to leave.
- **Contract type** is the most impactful categorical feature — month-to-month contracts drive the majority of churn.
- **TechSupport** and **InternetService** also show strong associations with churn rate.
- All plots have been saved to `outputs/eda/` for display in the Streamlit dashboard.